# Vision Editor — Jupyter Notebook
## Advanced Image Enhancement & Analysis

> **Domain**: Computer Vision / Digital Image Processing  
> **AI Task**: Face Detection (Haar Cascade — OpenCV pre-trained model)

---
### Introduction

Image processing is the backbone of modern Computer Vision (CV) pipelines.  
Low-quality inputs — noisy, dark, or low-contrast images — severely degrade the performance of AI models.  
This project demonstrates:
1. **Task One**: A full GUI editor built with `Tkinter` + `OpenCV` for real-time image manipulation.
2. **Task Two**: A scientific experiment proving that preprocessing improves CV model accuracy.

**Techniques covered:**
- Point operations: Brightness, Contrast, Exposure, Highlights, Shadows
- Geometric: Zoom (Bilinear / Nearest-Neighbor), Rotation
- Enhancement: Histogram Equalization, Gamma Correction
- Filtering: Laplacian, Sobel, Averaging, Median, Gaussian, Bilateral

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version:  {np.__version__}')

In [ ]:
# ── Utility: display helper ───────────────────────────────────────────
def show(images, titles, figsize=(16, 5), cmap=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2:  # grayscale
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_histogram(image, title='Histogram'):
    fig, ax = plt.subplots(figsize=(6, 2.5))
    colors = ('b', 'g', 'r')
    for i, c in enumerate(colors):
        hist = cv2.calcHist([image], [i], None, [256], [0, 256])
        ax.plot(hist, color=c, alpha=0.8)
    ax.set_title(title)
    ax.set_xlim(0, 255)
    plt.tight_layout()
    plt.show()

## Task One — Image Processing Operations

In [ ]:
# ── Load a test image ────────────────────────────────────────────────
# Change this path to your image or use the sample below
import urllib.request, os

SAMPLE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/640px-Gatto_europeo4.jpg'
SAMPLE_PATH = 'sample.jpg'

if not os.path.exists(SAMPLE_PATH):
    print('Downloading sample image...')
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)

img = cv2.imread(SAMPLE_PATH)
if img is None:
    img = np.random.randint(80, 180, (400, 600, 3), dtype=np.uint8)  # fallback
print(f'Image shape: {img.shape}')
show([img], ['Original'])
show_histogram(img, 'Original Histogram')

In [ ]:
# ── Point Operations ─────────────────────────────────────────────────
from image_processor import adjust_brightness, adjust_contrast, adjust_exposure

bright = adjust_brightness(img, 50)
dark   = adjust_brightness(img, -50)
hicon  = adjust_contrast(img, 60)
expo   = adjust_exposure(img, 50)

show([img, bright, dark, hicon, expo],
     ['Original', 'Brightness+50', 'Brightness-50', 'Contrast+60', 'Exposure+50'])

In [ ]:
# ── Geometric Operations ─────────────────────────────────────────────
from image_processor import rotate_image, zoom_image

rot45    = rotate_image(img, 45)
zoom_bl  = zoom_image(img, 1.5, 'Bilinear')
zoom_nn  = zoom_image(img, 1.5, 'Nearest')

show([img, rot45, zoom_bl, zoom_nn],
     ['Original', 'Rotate 45°', 'Zoom 1.5x Bilinear', 'Zoom 1.5x Nearest'])

In [ ]:
# ── Enhancement Operations ───────────────────────────────────────────
from image_processor import histogram_equalization, gamma_correction

histeq  = histogram_equalization(img)
gam05   = gamma_correction(img, 0.5)
gam20   = gamma_correction(img, 2.0)

show([img, histeq, gam05, gam20],
     ['Original', 'Hist. Equalization', 'Gamma 0.5', 'Gamma 2.0'])
show_histogram(img, 'Before Equalization')
show_histogram(histeq, 'After Equalization')

In [ ]:
# ── Filters (matrix convolutions) ────────────────────────────────────
from image_processor import apply_filter

filters = ['Laplacian', 'Sobel', 'Averaging', 'Median', 'Gaussian', 'Bilateral']
results = [apply_filter(img, f, ksize=5) for f in filters]

show([img] + results, ['Original'] + filters, figsize=(20, 4))

## Task Two — CV Accuracy Challenge
### Experiment: Face Detection on Degraded vs. Enhanced Image

In [ ]:
# ── Load a face image for the experiment ─────────────────────────────
# We'll use a portrait or a face detection sample.
# You can replace 'face.jpg' with any frontal face image.
face_img = img.copy()  # using same sample — replace for better demo

# ── Haar Cascade ─────────────────────────────────────────────────────
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
cascade      = cv2.CascadeClassifier(cascade_path)
print('Cascade loaded:', not cascade.empty())

In [ ]:
from cv_accuracy import run_accuracy_experiment

results = run_accuracy_experiment(face_img)

show([results['original_image'], results['degraded_image'], results['enh_annotated']],
     ['Original (Reference)',
      f"Degraded — Detected: {results['orig_faces']}",
      f"Enhanced — Detected: {results['enh_faces']}"])

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────
import pandas as pd

table = pd.DataFrame({
    'Metric':        ['Faces Detected', 'Accuracy (%)', 'Inference Speed (ms)', 'Overhead (ms)'],
    'Degraded':      [results['orig_faces'],
                      f"{results['orig_accuracy_pct']}%",
                      f"{results['orig_time_ms']} ms",
                      '—'],
    'Enhanced':      [results['enh_faces'],
                      f"{results['enh_accuracy_pct']}%",
                      f"{results['enh_time_ms']} ms",
                      f"{results['overhead_ms']} ms"],
})

print('\n=== CV Accuracy Comparison Table ===')
print(table.to_string(index=False))
print(f'\nConclusion: Enhancement improved accuracy by '
      f"{results['enh_accuracy_pct'] - results['orig_accuracy_pct']:.1f}% "
      f"with {results['overhead_ms']} ms overhead.")

In [ ]:
# ── Launch the GUI ────────────────────────────────────────────────────
print('To run the full GUI application, execute:')
print('   python main.py')